In [1]:
# script to generate filtered promoter bed file #

In [1]:
# import packages #
import pandas as pd
import pybedtools
from collections import Counter

In [5]:
# there are duplicates in the file at the HGNC name level #
# here we will:
# 1. xxxxxxx
#   # update: we can't control ties with bedtools which is messing up ordering, we need to do so from the command line. see first code block for that. 
#   # update update: we can use the combination of chrom_start_strand_hgnc as a unique ID for adding IDs
#   # as such: the first step is to open the fully annotated 1kb file, make that dictionary and add proceed to 2
# 2. replace the names in the BED files with the full ENSG_ID_ENST_ID_HGNC name - this will prevent collapsing on the same HGNC ID. all duplicates at the HGNC level are protein coding, ensembl canonical transcripts. this is how we get 18,761 total genes
# 3. confirm intervals are the same between the renamed files with bedtools
# 4. filter updated promoter files for exons for analysis/tabix filtering/etc.
# 5. save these final analysis level BEDs

In [142]:
# define a function for creating a list of keys and values from the fully annotated BED file
def build_the_dictionary (full_bed):
    # make a list for storing keys
    keys = []
    # iterate through the chrom, start, end, strand, and name in the full BED file
    for chrom, start, end, strand, name in zip(full_bed[0], full_bed[1], full_bed[2], full_bed[5], full_bed[3]):
        # check if plus strand - end doesn't change in promoter definitions
        if strand == '+':
            key = ('_').join([chrom, str(end), strand, name])
            keys.append(key)
        # check if minus strand - start doesn't change in promoter definitions 
        elif strand == '-':
            key = ('_').join([chrom, str(start), strand, name])
            keys.append(key)
    # check the length of the keys and make sure it equals the length of the bed file
    print(f'All keys are unique: {len(full_bed) == len(pd.Series(keys).unique())}')
    # build the dictionary
    dict2return = dict(zip(keys, full_bed[6]))
    return dict2return

In [146]:
# define a function for creating a list of keys and values from the fully annotated BED file
def build_the_keys (full_bed):
    # make a list for storing keys
    keys = []
    # iterate through the chrom, start, end, strand, and name in the full BED file
    for chrom, start, end, strand, name in zip(full_bed[0], full_bed[1], full_bed[2], full_bed[5], full_bed[3]):
        # check if plus strand - end doesn't change in promoter definitions
        if strand == '+':
            key = ('_').join([chrom, str(end), strand, name])
            keys.append(key)
        # check if minus strand - start doesn't change in promoter definitions 
        elif strand == '-':
            key = ('_').join([chrom, str(start), strand, name])
            keys.append(key)
    # check the length of the keys and make sure it equals the length of the bed file
    print(f'All keys are unique: {len(full_bed) == len(pd.Series(keys).unique())}')
    return keys

In [145]:
full_id_dict = build_the_dictionary(pd.read_csv('../raw_data/gencode.v44.protein.coding.canonical.autosomes.0.based.bed', sep = '\t', header = None))

All keys are unique: True


In [159]:
# open BED files and add keys for matching to full IDs
# 250 bp promoter
twoFifty_raw = pd.read_csv('../raw_data/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.093025.bed', sep = '\t', header = None)
# add keys
print('250 bp promoters')
twoFifty_raw.loc[:,'key2match'] = build_the_keys(twoFifty_raw)
# add full IDs
twoFifty_raw.loc[:, 'full_id'] = [full_id_dict.get(i) for i in twoFifty_raw['key2match']]
# 500 bp promoter
fiveHundred_raw = pd.read_csv('../raw_data/gencode.v44.protein.coding.500bp.promoters.autosomes.v2.093025.bed', sep = '\t', header = None)
# add keys
print('500 bp promoters')
fiveHundred_raw.loc[:,'key2match'] = build_the_keys(fiveHundred_raw)
# add full IDs
fiveHundred_raw.loc[:, 'full_id'] = [full_id_dict.get(i) for i in fiveHundred_raw['key2match']]
# 750 bp promoter
sevenFifty_raw = pd.read_csv('../raw_data/gencode.v44.protein.coding.750bp.promoters.autosomes.v2.093025.bed', sep = '\t', header = None)
# add keys
print('750 bp promoters')
sevenFifty_raw.loc[:, 'key2match'] = build_the_keys(sevenFifty_raw)
# add full IDs
sevenFifty_raw.loc[:, 'full_id'] = [full_id_dict.get(i) for i in sevenFifty_raw['key2match']]
# 1 kb promoters
oneKB_raw = pd.read_csv('../raw_data/gencode.v44.protein.coding.1kb.promoters.autosomes.v2.093025.bed', sep = '\t', header = None)
# add keys
print('1 kb promoters')
oneKB_raw.loc[:, 'key2match'] = build_the_keys(oneKB_raw)
# add full IDs
oneKB_raw.loc[:, 'full_id'] = [full_id_dict.get(i) for i in oneKB_raw['key2match']]

250 bp promoters
All keys are unique: True
500 bp promoters
All keys are unique: True
750 bp promoters
All keys are unique: True
1 kb promoters
All keys are unique: True


In [170]:
# reformat the BED files so that the full ID is now in the name column
# 1kb promoter
oneKB_full_id = pd.DataFrame({0 : oneKB_raw[0],
                              1 : oneKB_raw[1],
                              2 : oneKB_raw[2],
                              3 : oneKB_raw['full_id'],
                              4 : oneKB_raw[4],
                              5 : oneKB_raw[5]})
# 750bp promoter
sevenFifty_full_id = pd.DataFrame({0 : sevenFifty_raw[0],
                                   1 : sevenFifty_raw[1],
                                   2 : sevenFifty_raw[2],
                                   3 : sevenFifty_raw['full_id'],
                                   4 : sevenFifty_raw[4],
                                   5 : sevenFifty_raw[5]})
# 500bp promoter
fiveHundred_full_id = pd.DataFrame({0 : fiveHundred_raw[0],
                                    1 : fiveHundred_raw[1],
                                    2 : fiveHundred_raw[2],
                                    3 : fiveHundred_raw['full_id'],
                                    4 : fiveHundred_raw[4],
                                    5 : fiveHundred_raw[5]})
# 250bp promoter
twoFifty_full_id = pd.DataFrame({0 : twoFifty_raw[0],
                                 1 : twoFifty_raw[1],
                                 2 : twoFifty_raw[2],
                                 3 : twoFifty_raw['full_id'],
                                 4 : twoFifty_raw[4],
                                 5 : twoFifty_raw[5]})

In [176]:
# convert the newly annotated dataframes to BED files for sorting, subtracting, etc.
# 250bp promoters
twoFifty_bedtool = pybedtools.BedTool.from_dataframe(twoFifty_full_id).sort()
# 500bp promoters
fiveHundred_bedtool = pybedtools.BedTool.from_dataframe(fiveHundred_full_id).sort()
# 750bp promoters
sevenFifty_bedtool = pybedtools.BedTool.from_dataframe(sevenFifty_full_id).sort()
# oneKB promoters
oneKB_bedtool = pybedtools.BedTool.from_dataframe(oneKB_full_id)

In [191]:
# save the full, sorted, full ID to disk for using in other places
# 250bp promoters
twoFifty_bedtool.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.full.ID.bed', sep = '\t', index = False, header = False)
# 500bp promoters
fiveHundred_bedtool.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.500bp.promoters.autosomes.v2.full.ID.bed', sep = '\t', index = False, header = False)
# 750bp promoters
sevenFifty_bedtool.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.750bp.promoters.autosomes.v2.full.ID.bed', sep = '\t', index = False, header = False)
# 1kb promoters
oneKB_bedtool.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.1kb.promoters.autosomes.v2.full.ID.bed', sep = '\t', index = False, header = False)

In [186]:
# open the exon + splice junctions BED file from Stephen - 093025 #
exon_plus_splice = pybedtools.BedTool('../raw_data/gencode.v44.basic.annotation.exons.splice.autosomes.v2.093025.bed').sort()

In [195]:
# intersect the exons with the promoters to remove those sites overlapping exons #
# 250bp promoters
twoFifty_exon_filtered = twoFifty_bedtool.subtract(exon_plus_splice, s=True)
# 500bp promoters
fiveHundred_exon_filted = fiveHundred_bedtool.subtract(exon_plus_splice, s=True)
# 750bp promoters
sevenFifty_exon_filted = sevenFifty_bedtool.subtract(exon_plus_splice, s=True)
# 1 kb promoters 
oneKB_exon_filtered = oneKB_bedtool.subtract(exon_plus_splice, s=True)


In [199]:
# check the number of promoters remaining after filtering
# 1kb - this is the one from the methods, should equal 18,658
print('1Kb promoters')
print(f'There are {len(oneKB_exon_filtered.to_dataframe()['name'].unique())} promoters remaining after filtering for exons')
# 750bp promoters
print('750bp promoters')
print(f'There are {len(sevenFifty_exon_filted.to_dataframe()['name'].unique())} promoters remaining after filtering for exons')
# 500bp promoters
print('500bp promoters')
print(f'There are {len(fiveHundred_exon_filted.to_dataframe()['name'].unique())} promoters remaining after filtering for exons')
# 250bp promoters
print('250bp promoters')
print(f'There are {len(twoFifty_exon_filtered.to_dataframe()['name'].unique())} promoters remaining after fitlering for exons')

1Kb promoters
There are 18658 promoters remaining after filtering for exons
750bp promoters
There are 18604 promoters remaining after filtering for exons
500bp promoters
There are 18467 promoters remaining after filtering for exons
250bp promoters
There are 17958 promoters remaining after fitlering for exons


In [200]:
# there are the correct number of promoters post filtering per the methods for the the 1kb regions. save all to disk and move forward to filtering sat mut predictions for promoterAI, PARM, etc. comparisons #
# 250bp exon filtered promoters
twoFifty_exon_filtered.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.full.ID.exon.filtered.bed', sep = '\t', index = False, header = False)
# 500bp exon filtered promoters
fiveHundred_exon_filted.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.500bp.promoters.autosomes.v2.full.ID.exon.filtered.bed', sep = '\t', index = False, header = False)
# 750bp exon filtered promoters
sevenFifty_exon_filted.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.750bp.promoters.autosomes.v2.full.ID.exon.filtered.bed', sep = '\t', index = False, header = False)
# 1kb exon filtered promoters
oneKB_exon_filtered.to_dataframe().to_csv('../raw_data/reformatted_bed_files/gencode.v44.protein.coding.1kb.promoters.autosomes.v2.full.ID.exon.filtered.bed', sep = '\t', index = False, header = False)